# Fix Decimal Columns for Fabric Graph

Fabric Graph **does not support the `Decimal` data type** in lakehouse delta tables.
Queries return `null` for any column stored as `decimal` regardless of the ontology mapping.

This notebook casts all `decimal` columns to `double` so that Fabric Graph can read them.

**Prerequisites:**
- Attach this notebook to your target **Lakehouse** before running
- The tables referenced below must exist in the attached lakehouse

In [ ]:
# =================================================================
# Configuration — set your table names here
# =================================================================
# List the tables you want to fix. Leave empty [] to scan ALL tables.
TABLES_TO_FIX = []  # e.g. ["Trip", "Payment"]  or [] for all

# Target type for decimal columns ("double" recommended for Fabric Graph)
TARGET_TYPE = "double"

In [ ]:
# =================================================================
# Step 1: Discover tables and detect decimal columns
# =================================================================
from pyspark.sql import functions as F
import os

# Get attached lakehouse Tables path
tables_path = "Tables"

if TABLES_TO_FIX:
    table_names = TABLES_TO_FIX
else:
    # Auto-discover all delta tables
    table_names = [t.name for t in spark.catalog.listTables() if not t.isTemporary]
    print(f"Discovered {len(table_names)} tables: {table_names}")

# Scan each table for decimal columns
decimal_report = {}
for tname in table_names:
    try:
        df = spark.read.format("delta").load(f"{tables_path}/{tname}")
        decimal_cols = [
            (f.name, f.dataType.simpleString())
            for f in df.schema.fields
            if "decimal" in f.dataType.simpleString().lower()
        ]
        if decimal_cols:
            decimal_report[tname] = decimal_cols
            print(f"  {tname}: {len(decimal_cols)} decimal columns")
            for col_name, col_type in decimal_cols:
                print(f"    - {col_name}: {col_type}")
        else:
            print(f"  {tname}: no decimal columns (OK)")
    except Exception as e:
        print(f"  {tname}: ERROR reading table — {e}")

if not decimal_report:
    print("\n✓ No decimal columns found — nothing to fix!")
else:
    print(f"\n⚠ Found decimal columns in {len(decimal_report)} table(s)")

In [ ]:
# =================================================================
# Step 2: Preview schema changes (dry run)
# =================================================================
if decimal_report:
    print("Schema changes to apply:")
    print("=" * 60)
    for tname, cols in decimal_report.items():
        print(f"\nTable: {tname}")
        for col_name, col_type in cols:
            print(f"  {col_name}: {col_type} → {TARGET_TYPE}")
    print("\n" + "=" * 60)
    print(f"Total: {sum(len(c) for c in decimal_report.values())} columns in {len(decimal_report)} tables")
    print("\nRun the next cell to apply these changes.")
else:
    print("Nothing to preview — no decimal columns found.")

In [ ]:
# =================================================================
# Step 3: Apply the fix — cast decimal columns to target type
# =================================================================
if not decimal_report:
    print("Nothing to fix.")
else:
    for tname, cols in decimal_report.items():
        print(f"\nFixing {tname} ...")
        path = f"{tables_path}/{tname}"
        df = spark.read.format("delta").load(path)

        # Cast each decimal column
        for col_name, _ in cols:
            df = df.withColumn(col_name, F.col(col_name).cast(TARGET_TYPE))
            print(f"  ✓ {col_name} → {TARGET_TYPE}")

        # Overwrite the table with corrected schema
        df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .save(path)
        print(f"  ✓ {tname} saved successfully")

    print(f"\n✓ Done! Fixed {sum(len(c) for c in decimal_report.values())} columns in {len(decimal_report)} table(s).")

In [ ]:
# =================================================================
# Step 4: Verify — re-read tables and confirm no decimal columns
# =================================================================
print("Verification:")
all_ok = True
for tname in decimal_report.keys():
    df = spark.read.format("delta").load(f"{tables_path}/{tname}")
    remaining = [
        f.name for f in df.schema.fields
        if "decimal" in f.dataType.simpleString().lower()
    ]
    if remaining:
        print(f"  ✗ {tname}: still has decimal columns: {remaining}")
        all_ok = False
    else:
        print(f"  ✓ {tname}: all columns OK")
        # Show updated schema
        for f in df.schema.fields:
            print(f"    {f.name}: {f.dataType.simpleString()}")

if all_ok:
    print("\n✓ All tables verified — ready for Fabric Graph!")
    print("\nNext steps:")
    print("  1. The Direct Lake Semantic Model should auto-detect the schema change")
    print("  2. Re-create the ontology with: fabric-iq create --verify-lakehouse ...")
else:
    print("\n✗ Some columns still have decimal type — check the errors above.")